In [11]:
import pandas as pd

df = pd.read_excel("/content/spanish_economic_news_ibex_2026.xlsx", sheet_name="Articles")

df.shape

(337, 6)

In [12]:
# Fill article information down through its paragraph rows
df["Date"] = df["Date"].ffill()
df["Source"] = df["Source"].ffill()
df["Title"] = df["Title"].ffill()
df["URL"] = df["URL"].ffill()

# Combine all paragraphs belonging to each article
clean_df = (
    df.groupby(["Date", "Source", "Title", "URL"], sort=False)["Article Text"]
      .apply(lambda x: " ".join(x.dropna().astype(str)))
      .reset_index()
)

clean_df["Sentiment"] = None

clean_df.shape

(18, 6)

In [14]:
clean_df[["Date", "Title", "Article Text"]].head()

,Date,Title,Article Text
0,2026-06-10,"La Bolsa cede ante el cóctel de guerra, inflac...",Un dato menos negativo ha bastado para frenar ...
1,2026-06-16,"El Ibex sube un 0,7%, revalida nuevos máximos ...",Los mercados prolongan este martes el alivio i...
2,2026-06-19,"El Ibex gana un 3% en la semana, entre el aliv...",Las Bolsas mundiales han cerrado una semana ma...
3,2026-06-22,"El precio del petróleo baja a los 77 dólares, ...",Los inversores siguen de cerca los avances en ...
4,2026-06-23,El mercado refuerza la expectativa de alzas de...,"La última reunión de la Reserva Federal, la pr..."


In [ ]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis"
)

print("Spanish sentiment model loaded.")

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

Spanish sentiment model loaded.


In [ ]:
sentiment_model("La economía española mejora y las empresas aumentan sus beneficios.")

[{'label': 'POS', 'score': 0.7027977108955383}]

In [15]:
def convert_sentiment(result):
    label = result["label"]
    score = result["score"]

    if label == "POS":
        return 1.0 if score >= 0.75 else 0.5
    elif label == "NEG":
        return -1.0 if score >= 0.75 else -0.5
    else:
        return 0.0


results = sentiment_model(
    clean_df["Article Text"].tolist(),
    truncation=True
)

clean_df["Model Label"] = [r["label"] for r in results]
clean_df["Model Confidence"] = [r["score"] for r in results]
clean_df["Sentiment"] = [convert_sentiment(r) for r in results]

clean_df[[
    "Date",
    "Title",
    "Model Label",
    "Model Confidence",
    "Sentiment"
]]

NameError: name 'sentiment_model' is not defined

In [16]:
clean_df[[
    "Date",
    "Title",
    "Model Label",
    "Model Confidence",
    "Sentiment"
]].to_string(index=False)

KeyError: "['Model Label', 'Model Confidence'] not in index"

In [ ]:
from transformers import pipeline

finance_sentiment_model = pipeline(
    "sentiment-analysis",
    model="bardsai/finance-sentiment-es-base"
)

print("Spanish financial sentiment model loaded.")

config.json:   0%|          | 0.00/928 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  439MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/730k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Spanish financial sentiment model loaded.


In [ ]:
finance_sentiment_model(
    "El Ibex sube un 3% y alcanza nuevos máximos."
)


[{'label': 'positive', 'score': 0.9999426603317261}]

In [9]:
def convert_finance_sentiment(result):
    label = result["label"].lower()
    score = result["score"]

    if label == "positive":
        return 1.0 if score >= 0.75 else 0.5
    elif label == "negative":
        return -1.0 if score >= 0.75 else -0.5
    else:
        return 0.0


finance_results = finance_sentiment_model(
    clean_df["Article Text"].tolist(),
    truncation=True
)

clean_df["Finance Label"] = [r["label"] for r in finance_results]
clean_df["Finance Confidence"] = [r["score"] for r in finance_results]
clean_df["Sentiment"] = [
    convert_finance_sentiment(r) for r in finance_results
]

clean_df[[
    "Date",
    "Title",
    "Finance Label",
    "Finance Confidence",
    "Sentiment"
]]

NameError: name 'finance_sentiment_model' is not defined

In [8]:
clean_df[[
    "Date",
    "Title",
    "Finance Label",
    "Finance Confidence"
]].to_string(index=False)

KeyError: "['Finance Label', 'Finance Confidence'] not in index"

In [17]:
# Rule-based Spanish financial sentiment: Step 1
# Define transparent positive and negative financial terms.

positive_terms = {
    "sube": 1,
    "subida": 1,
    "subidas": 1,
    "avance": 1,
    "avanza": 1,
    "ganancia": 1,
    "ganancias": 1,
    "crecimiento": 1,
    "mejora": 1,
    "mejora": 1,
    "recuperación": 1,
    "optimismo": 1,
    "fortaleza": 1,
    "impulso": 1,
    "alivio": 1,
    "récord": 1,
    "máximo": 1,
    "máximos": 1,
    "supera": 1,
    "consolida": 1,
    "favorable": 1,
    "positivo": 1,
    "positiva": 1,
    "positivos": 1,
    "positivas": 1,
}

negative_terms = {
    "cae": -1,
    "caída": -1,
    "caídas": -1,
    "cede": -1,
    "descenso": -1,
    "descensos": -1,
    "pérdida": -1,
    "pérdidas": -1,
    "inflación": -1,
    "guerra": -1,
    "riesgo": -1,
    "riesgos": -1,
    "dudas": -1,
    "volatilidad": -1,
    "crisis": -1,
    "temor": -1,
    "miedo": -1,
    "empeora": -1,
    "debilidad": -1,
    "retroceso": -1,
    "presión": -1,
    "amenaza": -1,
    "negativo": -1,
    "negativa": -1,
    "negativos": -1,
    "negativas": -1,
}

In [18]:
# Rule-based Spanish financial sentiment: Step 2
# Contextual phrases with stronger meaning than individual words.

positive_phrases = {
    "el ibex sube": 2,
    "el ibex gana": 2,
    "el ibex avanza": 2,
    "el ibex alcanza máximos": 2,
    "el ibex marca máximos": 2,
    "el ibex consolida": 2,
    "las bolsas suben": 2,
    "las bolsas avanzan": 2,
    "los mercados suben": 2,
    "los mercados avanzan": 2,
    "se reducen las dudas": 2,
    "disminuyen las dudas": 2,
    "se reduce el riesgo": 2,
    "baja la inflación": 2,
    "cae la inflación": 2,
    "baja el petróleo": 1,
    "cae el petróleo": 1,
    "alivio en los mercados": 2,
    "apetito por el riesgo": 1,
}

negative_phrases = {
    "el ibex cae": -2,
    "el ibex cede": -2,
    "el ibex pierde": -2,
    "el ibex baja": -2,
    "las bolsas caen": -2,
    "las bolsas ceden": -2,
    "los mercados caen": -2,
    "los mercados ceden": -2,
    "aumentan las dudas": -2,
    "crecen las dudas": -2,
    "aumenta el riesgo": -2,
    "aumentan los riesgos": -2,
    "sube la inflación": -2,
    "aumenta la inflación": -2,
    "se dispara la inflación": -2,
    "sube el petróleo": -1,
    "se dispara el petróleo": -2,
    "presión sobre los mercados": -2,
    "aversión al riesgo": -2,
}

In [19]:
# Rule-based Spanish financial sentiment: Step 3 (revised)
# Gives greater weight to the headline and limits repeated words.

import re
import pandas as pd


def normalize_text(text):
    """Lowercase and normalize whitespace."""
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def find_phrase_hits(text, phrase_dict):
    """Find contextual phrase matches."""
    hits = []

    for phrase, weight in phrase_dict.items():
        if phrase in text:
            hits.append((phrase, weight))

    return hits


def find_word_hits(text, term_dict):
    """
    Find individual word matches.
    Each term contributes at most once, regardless of repetition.
    """
    hits = []

    for term, weight in term_dict.items():
        if re.search(r"\b" + re.escape(term) + r"\b", text):
            hits.append((term, weight))

    return hits


def rule_based_sentiment(title, article_text):

    title = normalize_text(title)
    article_text = normalize_text(article_text)

    # Combine title + article for phrase/word detection
    full_text = title + " " + article_text

    raw_score = 0
    positive_hits = []
    negative_hits = []

    # -----------------------------------------
    # 1. Headline receives stronger weighting
    # -----------------------------------------

    headline_positive = find_phrase_hits(title, positive_phrases)
    headline_negative = find_phrase_hits(title, negative_phrases)

    for phrase, weight in headline_positive:
        raw_score += weight * 2
        positive_hits.append("HEADLINE: " + phrase)

    for phrase, weight in headline_negative:
        raw_score += weight * 2
        negative_hits.append("HEADLINE: " + phrase)


    # -----------------------------------------
    # 2. Individual headline terms
    # -----------------------------------------

    for term, weight in find_word_hits(title, positive_terms):
        raw_score += weight * 2
        positive_hits.append("HEADLINE: " + term)

    for term, weight in find_word_hits(title, negative_terms):
        raw_score += weight * 2
        negative_hits.append("HEADLINE: " + term)


    # -----------------------------------------
    # 3. Contextual phrases in full article
    # -----------------------------------------

    for phrase, weight in find_phrase_hits(full_text, positive_phrases):
        raw_score += weight
        positive_hits.append(phrase)

    for phrase, weight in find_phrase_hits(full_text, negative_phrases):
        raw_score += weight
        negative_hits.append(phrase)


    # -----------------------------------------
    # 4. Individual terms in full article
    # -----------------------------------------

    for term, weight in find_word_hits(full_text, positive_terms):
        raw_score += weight
        positive_hits.append(term)

    for term, weight in find_word_hits(full_text, negative_terms):
        raw_score += weight
        negative_hits.append(term)


    # -----------------------------------------
    # 5. Convert raw score to final scale
    # -----------------------------------------

    if raw_score >= 8:
        sentiment = 1.0
    elif raw_score >= 3:
        sentiment = 0.5
    elif raw_score <= -8:
        sentiment = -1.0
    elif raw_score <= -3:
        sentiment = -0.5
    else:
        sentiment = 0.0


    return pd.Series({
        "Rule Raw Score": raw_score,
        "Rule Sentiment": sentiment,
        "Positive Hits": ", ".join(sorted(set(positive_hits))),
        "Negative Hits": ", ".join(sorted(set(negative_hits)))
    })

In [20]:
# Apply the rule-based sentiment detector to all 18 articles

rule_results = clean_df.apply(
    lambda row: rule_based_sentiment(
        row["Title"],
        row["Article Text"]
    ),
    axis=1
)

rule_df = pd.concat(
    [
        clean_df[["Date", "Title"]],
        rule_results
    ],
    axis=1
)

# Display the results
display(rule_df)

,Date,Title,Rule Raw Score,Rule Sentiment,Positive Hits,Negative Hits
0,2026-06-10,"La Bolsa cede ante el cóctel de guerra, inflac...",-10,-1.0,"apetito por el riesgo, avanza, crecimiento, im...","HEADLINE: cede, HEADLINE: guerra, HEADLINE: in..."
1,2026-06-16,"El Ibex sube un 0,7%, revalida nuevos máximos ...",17,1.0,"HEADLINE: el ibex sube, HEADLINE: máximos, HEA...","cae, caída, caídas, descenso, descensos, infla..."
2,2026-06-19,"El Ibex gana un 3% en la semana, entre el aliv...",6,0.5,"HEADLINE: alivio, HEADLINE: el ibex gana, aliv...","caída, caídas, dudas, guerra, inflación, riesg..."
3,2026-06-22,"El precio del petróleo baja a los 77 dólares, ...",-7,-0.5,"máximo, subida, subidas","HEADLINE: guerra, cede, dudas, guerra, inflaci..."
4,2026-06-23,El mercado refuerza la expectativa de alzas de...,4,0.5,"HEADLINE: máximos, alivio, fortaleza, impulso,...","cae, dudas, guerra, inflación"
5,2026-06-29,El Ibex se aleja de sus máximos en un arranque...,-4,-0.5,"HEADLINE: máximos, avanza, ganancias, mejora, ...","HEADLINE: dudas, caída, caídas, cede, dudas, p..."
6,2026-07-02,El Ibex toca nuevos máximos al desinflarse las...,7,0.5,"HEADLINE: máximos, HEADLINE: subida, alivio, a...","aumentan las dudas, cae, descenso, dudas, infl..."
7,2026-07-08,Sesión negra al anunciar Trump el final del al...,-9,-1.0,"sube, subida","aumenta el riesgo, caída, caídas, crisis, guer..."
8,2026-07-10,El Ibex registra su peor semana en dos meses m...,1,0.0,"HEADLINE: impulso, crecimiento, ganancias, imp...","amenaza, caída, caídas, debilidad, dudas, guer..."
9,2026-07-17,Los inversores cuestionan las valoraciones de ...,-3,-0.5,"avance, avanza, crecimiento, ganancias, mejora...","caída, caídas, cede, crisis, debilidad, descen..."


In [21]:
# Rule-based Spanish financial sentiment: Step 4
# Explicit financial-event phrases.
#
# These rules are stronger than generic words because they capture
# the financial meaning of the event being reported.

financial_positive_phrases = {
    "el ibex sube": 3,
    "el ibex gana": 3,
    "el ibex avanza": 3,
    "el ibex rebota": 3,
    "el ibex marca nuevos máximos": 4,
    "el ibex toca nuevos máximos": 4,
    "el ibex alcanza nuevos máximos": 4,
    "el ibex consolida": 3,
    "las bolsas suben": 3,
    "las bolsas avanzan": 3,
    "los mercados suben": 3,
    "los mercados avanzan": 3,
    "sube un": 2,
    "gana un": 2,
    "avanza un": 2,
    "nuevo máximo": 3,
    "nuevos máximos": 3,
    "máximos históricos": 4,
    "apetito por la bolsa": 2,
    "apetito por el riesgo": 2,
    "alivio en los mercados": 2,
    "se reduce el riesgo": 2,
    "se reducen las dudas": 2,
    "disminuyen las dudas": 2,
}

financial_negative_phrases = {
    "el ibex cae": -3,
    "el ibex cede": -3,
    "el ibex pierde": -3,
    "el ibex baja": -3,
    "el ibex retrocede": -3,
    "el ibex se hunde": -4,
    "el ibex marca mínimos": -4,
    "las bolsas caen": -3,
    "las bolsas ceden": -3,
    "los mercados caen": -3,
    "los mercados ceden": -3,
    "cae un": -2,
    "pierde un": -2,
    "retrocede un": -2,
    "peor sesión": -4,
    "peor semana": -4,
    "sesión negra": -4,
    "se dispara el petróleo": -3,
    "sube el petróleo": -2,
    "aumenta la inflación": -3,
    "sube la inflación": -3,
    "aumentan las dudas": -2,
    "crecen las dudas": -2,
    "aumenta el riesgo": -2,
    "aumentan los riesgos": -2,
    "aversión al riesgo": -2,
    "presión sobre los mercados": -2,
}

In [22]:
# Rule-based Spanish financial sentiment: Step 5
# Event-based detector.
#
# Financial event phrases are the main signal.
# Generic body words are deliberately NOT counted.

def event_based_sentiment(title, article_text):

    title = normalize_text(title)
    article_text = normalize_text(article_text)
    full_text = title + " " + article_text

    raw_score = 0
    positive_hits = []
    negative_hits = []

    # -----------------------------------------
    # 1. Financial event phrases in headline
    #    receive double weight
    # -----------------------------------------

    for phrase, weight in financial_positive_phrases.items():
        if phrase in title:
            raw_score += weight * 2
            positive_hits.append("HEADLINE: " + phrase)

    for phrase, weight in financial_negative_phrases.items():
        if phrase in title:
            raw_score += weight * 2
            negative_hits.append("HEADLINE: " + phrase)

    # -----------------------------------------
    # 2. Financial event phrases in article
    # -----------------------------------------

    for phrase, weight in financial_positive_phrases.items():
        if phrase in full_text:
            raw_score += weight
            positive_hits.append(phrase)

    for phrase, weight in financial_negative_phrases.items():
        if phrase in full_text:
            raw_score += weight
            negative_hits.append(phrase)

    # -----------------------------------------
    # 3. Convert raw score to five-point scale
    # -----------------------------------------

    if raw_score >= 8:
        sentiment = 1.0
    elif raw_score >= 3:
        sentiment = 0.5
    elif raw_score <= -8:
        sentiment = -1.0
    elif raw_score <= -3:
        sentiment = -0.5
    else:
        sentiment = 0.0

    return pd.Series({
        "Event Raw Score": raw_score,
        "Event Sentiment": sentiment,
        "Positive Events": ", ".join(sorted(set(positive_hits))),
        "Negative Events": ", ".join(sorted(set(negative_hits)))
    })

In [23]:
# Apply the event-based detector to all 18 articles

event_results = clean_df.apply(
    lambda row: event_based_sentiment(
        row["Title"],
        row["Article Text"]
    ),
    axis=1
)

event_df = pd.concat(
    [
        clean_df[["Date", "Title"]],
        event_results
    ],
    axis=1
)

display(event_df)

,Date,Title,Event Raw Score,Event Sentiment,Positive Events,Negative Events
0,2026-06-10,"La Bolsa cede ante el cóctel de guerra, inflac...",0,0.0,apetito por el riesgo,aumentan las dudas
1,2026-06-16,"El Ibex sube un 0,7%, revalida nuevos máximos ...",33,1.0,"HEADLINE: el ibex sube, HEADLINE: nuevos máxim...",
2,2026-06-19,"El Ibex gana un 3% en la semana, entre el aliv...",15,1.0,"HEADLINE: el ibex gana, HEADLINE: gana un, el ...",pierde un
3,2026-06-22,"El precio del petróleo baja a los 77 dólares, ...",3,0.5,nuevo máximo,
4,2026-06-23,El mercado refuerza la expectativa de alzas de...,0,0.0,,
5,2026-06-29,El Ibex se aleja de sus máximos en un arranque...,9,1.0,"avanza un, máximos históricos, nuevos máximos",
6,2026-07-02,El Ibex toca nuevos máximos al desinflarse las...,25,1.0,"HEADLINE: el ibex toca nuevos máximos, HEADLIN...",aumentan las dudas
7,2026-07-08,Sesión negra al anunciar Trump el final del al...,-26,-1.0,,"HEADLINE: peor sesión, HEADLINE: sesión negra,..."
8,2026-07-10,El Ibex registra su peor semana en dos meses m...,-8,-1.0,máximos históricos,"HEADLINE: peor semana, peor semana"
9,2026-07-17,Los inversores cuestionan las valoraciones de ...,-4,-0.5,,peor sesión


In [24]:
import pandas as pd

# Load the Excel file
df = pd.read_excel("spanish_economic_news_ibex_2026.xlsx")

# Restore article metadata across paragraph rows
df["Date"] = df["Date"].ffill()
df["Source"] = df["Source"].ffill()
df["Title"] = df["Title"].ffill()
df["URL"] = df["URL"].ffill()

# Recombine paragraph rows into one record per article
clean_df = (
    df.groupby(["Date", "Source", "Title", "URL"], sort=False)["Article Text"]
      .apply(lambda x: " ".join(x.dropna().astype(str)))
      .reset_index()
)

# Add empty sentiment column
clean_df["Sentiment"] = None

print("Number of articles:", len(clean_df))

Number of articles: 18


In [25]:
import os

print(os.listdir("/content"))

['.config', 'spanish_economic_news_ibex_2026.xlsx', 'sample_data']


In [24]:
from google.colab import files

uploaded = files.upload()

Saving spanish_economic_news_ibex_2026.xlsx to spanish_economic_news_ibex_2026 (1).xlsx


In [26]:
# Rule-based Spanish financial sentiment: Step 6
# Headline-first rules.
#
# The headline is the primary signal.
# The article body will only be used when the headline is ambiguous.

headline_positive_rules = {
    "el ibex sube": 3,
    "el ibex gana": 3,
    "el ibex avanza": 3,
    "el ibex rebota": 3,
    "el ibex toca nuevos máximos": 3,
    "el ibex marca nuevos máximos": 3,
    "el ibex alcanza nuevos máximos": 3,
    "el ibex consolida": 3,
    "nuevos máximos": 2,
    "máximos históricos": 3,
    "sube un": 2,
    "gana un": 2,
    "avanza un": 2,
    "apetito por la bolsa": 2,
    "apetito por el riesgo": 2,
    "alivio": 1,
    "optimismo": 1,
    "crecimiento": 1,
    "recuperación": 1,
}

headline_negative_rules = {
    "el ibex cae": -3,
    "el ibex cede": -3,
    "el ibex pierde": -3,
    "el ibex baja": -3,
    "el ibex retrocede": -3,
    "el ibex se hunde": -4,
    "se aleja de sus máximos": -3,
    "se paraliza": -2,
    "peor sesión": -4,
    "peor semana": -4,
    "sesión negra": -4,
    "cae un": -2,
    "pierde un": -2,
    "retrocede un": -2,
    "aumentan las dudas": -2,
    "crecen las dudas": -2,
    "aumenta el riesgo": -2,
    "aumentan los riesgos": -2,
    "aversión al riesgo": -2,
    "guerra": -1,
    "inflación": -1,
    "amenaza": -1,
}

In [27]:
# Rule-based Spanish financial sentiment: Step 7
# Headline-first scoring function.

def headline_first_sentiment(title, article_text):

    title = normalize_text(title)
    article_text = normalize_text(article_text)

    raw_score = 0
    positive_hits = []
    negative_hits = []

    # -----------------------------------------
    # 1. Check positive headline rules
    # -----------------------------------------

    for phrase, weight in headline_positive_rules.items():
        if phrase in title:
            raw_score += weight
            positive_hits.append(phrase)

    # -----------------------------------------
    # 2. Check negative headline rules
    # -----------------------------------------

    for phrase, weight in headline_negative_rules.items():
        if phrase in title:
            raw_score += weight
            negative_hits.append(phrase)

    # -----------------------------------------
    # 3. Special handling for mixed headlines
    # -----------------------------------------

    # If the headline explicitly says the IBEX rises/gains,
    # give that direct market movement priority over generic
    # negative context such as "dudas" or "inflación".

    if (
        ("el ibex sube" in title or
         "el ibex gana" in title or
         "el ibex avanza" in title or
         "el ibex rebota" in title)
        and raw_score > 0
    ):
        raw_score = max(raw_score, 3)

    # If the headline explicitly says the IBEX falls/cedes,
    # give that direct market movement priority.

    if (
        ("el ibex cae" in title or
         "el ibex cede" in title or
         "el ibex pierde" in title or
         "el ibex baja" in title or
         "el ibex retrocede" in title)
        and raw_score < 0
    ):
        raw_score = min(raw_score, -3)

    # -----------------------------------------
    # 4. Convert to five-point sentiment scale
    # -----------------------------------------

    if raw_score >= 5:
        sentiment = 1.0
    elif raw_score >= 2:
        sentiment = 0.5
    elif raw_score <= -5:
        sentiment = -1.0
    elif raw_score <= -2:
        sentiment = -0.5
    else:
        sentiment = 0.0

    return pd.Series({
        "Headline Raw Score": raw_score,
        "Headline Sentiment": sentiment,
        "Positive Headline Signals": ", ".join(positive_hits),
        "Negative Headline Signals": ", ".join(negative_hits)
    })

In [28]:
# Apply the headline-first detector to all 18 articles

headline_results = clean_df.apply(
    lambda row: headline_first_sentiment(
        row["Title"],
        row["Article Text"]
    ),
    axis=1
)

headline_df = pd.concat(
    [
        clean_df[["Date", "Title"]],
        headline_results
    ],
    axis=1
)

display(headline_df)

,Date,Title,Headline Raw Score,Headline Sentiment,Positive Headline Signals,Negative Headline Signals
0,2026-06-10,"La Bolsa cede ante el cóctel de guerra, inflac...",-2,-0.5,,"guerra, inflación"
1,2026-06-16,"El Ibex sube un 0,7%, revalida nuevos máximos ...",7,1.0,"el ibex sube, nuevos máximos, sube un",
2,2026-06-19,"El Ibex gana un 3% en la semana, entre el aliv...",6,1.0,"el ibex gana, gana un, alivio",
3,2026-06-22,"El precio del petróleo baja a los 77 dólares, ...",-1,0.0,,guerra
4,2026-06-23,El mercado refuerza la expectativa de alzas de...,0,0.0,,
5,2026-06-29,El Ibex se aleja de sus máximos en un arranque...,-3,-0.5,,se aleja de sus máximos
6,2026-07-02,El Ibex toca nuevos máximos al desinflarse las...,5,1.0,"el ibex toca nuevos máximos, nuevos máximos",
7,2026-07-08,Sesión negra al anunciar Trump el final del al...,-8,-1.0,,"peor sesión, sesión negra"
8,2026-07-10,El Ibex registra su peor semana en dos meses m...,-4,-0.5,,peor semana
9,2026-07-17,Los inversores cuestionan las valoraciones de ...,0,0.0,,


In [29]:
# Step 3M — Additional contextual financial headline rules

headline_context_rules = {
    # Positive market/economic implications
    "el precio del petróleo baja": 2,
    "el petróleo baja": 2,
    "el petróleo cae": 2,
    "alivio": 1,

    # Negative market/economic implications
    "expectativa de alzas de tipos": -2,
    "expectativa de subida de tipos": -2,
    "expectativas de alzas de tipos": -2,
    "expectativas de subida de tipos": -2,
    "cuestionan las valoraciones": -2,
    "bloquean el camino a nuevos máximos": -3,
    "ante las dudas": -1,
    "se paraliza tras": -1,

    # Explicit positive market event
    "subida del 466%": 3,
    "subida de un 466%": 3,
}

In [30]:
# Step 3N — Final candidate headline-first scoring function

def final_headline_sentiment(title, article_text):

    title = normalize_text(title)

    raw_score = 0
    positive_hits = []
    negative_hits = []

    # -----------------------------------------
    # 1. Main headline rules
    # -----------------------------------------

    for phrase, weight in headline_positive_rules.items():
        if phrase in title:
            raw_score += weight
            positive_hits.append(phrase)

    for phrase, weight in headline_negative_rules.items():
        if phrase in title:
            raw_score += weight
            negative_hits.append(phrase)

    # -----------------------------------------
    # 2. Context-specific financial rules
    # -----------------------------------------

    for phrase, weight in headline_context_rules.items():
        if phrase in title:
            raw_score += weight

            if weight > 0:
                positive_hits.append("CONTEXT: " + phrase)
            else:
                negative_hits.append("CONTEXT: " + phrase)

    # -----------------------------------------
    # 3. Direct IBEX movement takes priority
    # -----------------------------------------

    if (
        ("el ibex sube" in title or
         "el ibex gana" in title or
         "el ibex avanza" in title or
         "el ibex rebota" in title)
        and raw_score > 0
    ):
        raw_score = max(raw_score, 3)

    if (
        ("el ibex cae" in title or
         "el ibex cede" in title or
         "el ibex pierde" in title or
         "el ibex baja" in title or
         "el ibex retrocede" in title)
        and raw_score < 0
    ):
        raw_score = min(raw_score, -3)

    # -----------------------------------------
    # 4. Convert to five-point sentiment scale
    # -----------------------------------------

    if raw_score >= 5:
        sentiment = 1.0
    elif raw_score >= 2:
        sentiment = 0.5
    elif raw_score <= -5:
        sentiment = -1.0
    elif raw_score <= -2:
        sentiment = -0.5
    else:
        sentiment = 0.0

    return pd.Series({
        "Final Raw Score": raw_score,
        "Final Sentiment": sentiment,
        "Positive Signals": ", ".join(sorted(set(positive_hits))),
        "Negative Signals": ", ".join(sorted(set(negative_hits)))
    })

In [31]:
# Apply the final candidate detector to all 18 articles

final_results = clean_df.apply(
    lambda row: final_headline_sentiment(
        row["Title"],
        row["Article Text"]
    ),
    axis=1
)

final_df = pd.concat(
    [
        clean_df[["Date", "Title"]],
        final_results
    ],
    axis=1
)

display(final_df)

,Date,Title,Final Raw Score,Final Sentiment,Positive Signals,Negative Signals
0,2026-06-10,"La Bolsa cede ante el cóctel de guerra, inflac...",-2,-0.5,,"guerra, inflación"
1,2026-06-16,"El Ibex sube un 0,7%, revalida nuevos máximos ...",7,1.0,"el ibex sube, nuevos máximos, sube un",
2,2026-06-19,"El Ibex gana un 3% en la semana, entre el aliv...",7,1.0,"CONTEXT: alivio, alivio, el ibex gana, gana un",
3,2026-06-22,"El precio del petróleo baja a los 77 dólares, ...",3,0.5,"CONTEXT: el petróleo baja, CONTEXT: el precio ...",guerra
4,2026-06-23,El mercado refuerza la expectativa de alzas de...,-2,-0.5,,CONTEXT: expectativa de alzas de tipos
5,2026-06-29,El Ibex se aleja de sus máximos en un arranque...,-3,-0.5,,se aleja de sus máximos
6,2026-07-02,El Ibex toca nuevos máximos al desinflarse las...,5,1.0,"el ibex toca nuevos máximos, nuevos máximos",
7,2026-07-08,Sesión negra al anunciar Trump el final del al...,-8,-1.0,,"peor sesión, sesión negra"
8,2026-07-10,El Ibex registra su peor semana en dos meses m...,-4,-0.5,,peor semana
9,2026-07-17,Los inversores cuestionan las valoraciones de ...,-2,-0.5,,CONTEXT: cuestionan las valoraciones


In [32]:
# Lock the final sentiment scores into clean_df

clean_df["Sentiment"] = final_df["Final Sentiment"]

# Check the final dataset
display(clean_df[["Date", "Title", "Sentiment"]])

,Date,Title,Sentiment
0,2026-06-10,"La Bolsa cede ante el cóctel de guerra, inflac...",-0.5
1,2026-06-16,"El Ibex sube un 0,7%, revalida nuevos máximos ...",1.0
2,2026-06-19,"El Ibex gana un 3% en la semana, entre el aliv...",1.0
3,2026-06-22,"El precio del petróleo baja a los 77 dólares, ...",0.5
4,2026-06-23,El mercado refuerza la expectativa de alzas de...,-0.5
5,2026-06-29,El Ibex se aleja de sus máximos en un arranque...,-0.5
6,2026-07-02,El Ibex toca nuevos máximos al desinflarse las...,1.0
7,2026-07-08,Sesión negra al anunciar Trump el final del al...,-1.0
8,2026-07-10,El Ibex registra su peor semana en dos meses m...,-0.5
9,2026-07-17,Los inversores cuestionan las valoraciones de ...,-0.5


In [33]:
print(clean_df.columns.tolist())

['Date', 'Source', 'Title', 'URL', 'Article Text', 'Sentiment']


In [34]:
print(clean_df.loc[
    clean_df["Date"].astype(str) >= "2026-07-08",
    ["Date", "Title", "Sentiment"]
].sort_values("Date").to_string(index=False))

      Date                                                                                                                                    Title  Sentiment
2026-07-08 Sesión negra al anunciar Trump el final del alto el fuego: el petróleo se dispara a 80 dólares y el Ibex vive su peor sesión desde marzo       -1.0
2026-07-10                                           El Ibex registra su peor semana en dos meses mientras la IA mantiene el impulso de Wall Street       -0.5
2026-07-17                                                                  Los inversores cuestionan las valoraciones de la IA y agitan las Bolsas       -0.5
2026-07-21                                        La IA, la guerra en Oriente Próximo y la inflación bloquean el camino a nuevos máximos bursátiles       -0.5
2026-07-27                                                         La nueva fiebre de la IA tiene acento chino: CXMT debuta con una subida del 466%        0.5
2026-07-28                                    

In [35]:
print(clean_df[["Date", "Title", "Sentiment"]].to_string(index=False))


      Date                                                                                                                                    Title  Sentiment
2026-06-10                                                                   La Bolsa cede ante el cóctel de guerra, inflación y ajuste tecnológico       -0.5
2026-06-16                                                                 El Ibex sube un 0,7%, revalida nuevos máximos y supera los 19.100 puntos        1.0
2026-06-19                                                            El Ibex gana un 3% en la semana, entre el alivio y la cautela en los mercados        1.0
2026-06-22                                                    El precio del petróleo baja a los 77 dólares, niveles del inicio de la guerra en Irán        0.5
2026-06-23                                     El mercado refuerza la expectativa de alzas de tipos en EE UU y lleva al dólar a máximos de 12 meses       -0.5
2026-06-29                                    

In [36]:
ibex_df = pd.read_excel(
    "spanish_economic_news_ibex_2026.xlsx",
    sheet_name="IBEX"
)

print(ibex_df.head())
print(ibex_df.tail())
print(ibex_df.columns.tolist())

                  Date  IBEX 35 Close  Weekly Return  Converted Close
0  2026-10-06 00:00:00        18142.7            NaN           181427
1  2026-11-06 00:00:00        18290.1            NaN           182901
2  2026-12-06 00:00:00        18764.4       0.034267           187644
3           15/06/2026        19032.0            NaN            19032
4           16/06/2026        19163.6            NaN           191636
                   Date  IBEX 35 Close  Weekly Return  Converted Close
39  2026-04-08 00:00:00        20023.6            NaN           200236
40  2026-05-08 00:00:00        20057.0            NaN            20057
41  2026-06-08 00:00:00        20180.4            NaN           201804
42  2026-07-08 00:00:00        20176.0       0.009678            20176
43  2026-10-08 00:00:00        20173.0            NaN            20173
['Date', 'IBEX 35 Close', 'Weekly Return', 'Converted Close']


In [41]:
ibex_df["Date"] = pd.to_datetime(
    ibex_df["Date"],
    dayfirst=True,
    errors="coerce"
)

print(ibex_df[["Date", "IBEX 35 Close", "Weekly Return"]].head(10))
print(ibex_df[["Date", "IBEX 35 Close", "Weekly Return"]].tail(10))

        Date  IBEX 35 Close  Weekly Return
0 2026-10-06        18142.7            NaN
1 2026-11-06        18290.1            NaN
2 2026-12-06        18764.4       0.034267
3 2026-06-15        19032.0            NaN
4 2026-06-16        19163.6            NaN
5 2026-06-17        19421.9            NaN
6 2026-06-18        19404.1            NaN
7 2026-06-19        19347.4       0.016572
8 2026-06-22        19542.3            NaN
9 2026-06-23        19476.5            NaN
         Date  IBEX 35 Close  Weekly Return
34 2026-07-28        19727.0            NaN
35 2026-07-29        19412.7            NaN
36 2026-07-30        19757.7            NaN
37 2026-07-31        19782.9       0.002107
38 2026-03-08        19982.6            NaN
39 2026-04-08        20023.6            NaN
40 2026-05-08        20057.0            NaN
41 2026-06-08        20180.4            NaN
42 2026-07-08        20176.0       0.009678
43 2026-10-08        20173.0            NaN


In [42]:
# Fix the incorrectly formatted dates at the beginning
ibex_df.loc[0, "Date"] = pd.Timestamp("2026-06-10")
ibex_df.loc[1, "Date"] = pd.Timestamp("2026-06-11")
ibex_df.loc[2, "Date"] = pd.Timestamp("2026-06-12")

# Fix the incorrectly formatted dates in August
ibex_df.loc[38, "Date"] = pd.Timestamp("2026-08-03")
ibex_df.loc[39, "Date"] = pd.Timestamp("2026-08-04")
ibex_df.loc[40, "Date"] = pd.Timestamp("2026-08-05")
ibex_df.loc[41, "Date"] = pd.Timestamp("2026-08-06")
ibex_df.loc[42, "Date"] = pd.Timestamp("2026-08-07")
ibex_df.loc[43, "Date"] = pd.Timestamp("2026-08-10")

print(ibex_df[["Date", "IBEX 35 Close", "Weekly Return"]].to_string(index=False))

      Date  IBEX 35 Close  Weekly Return
2026-06-10        18142.7            NaN
2026-06-11        18290.1            NaN
2026-06-12        18764.4       0.034267
2026-06-15        19032.0            NaN
2026-06-16        19163.6            NaN
2026-06-17        19421.9            NaN
2026-06-18        19404.1            NaN
2026-06-19        19347.4       0.016572
2026-06-22        19542.3            NaN
2026-06-23        19476.5            NaN
2026-06-24        19389.5            NaN
2026-06-25        19513.6            NaN
2026-06-26        19425.3      -0.005987
2026-06-29        19387.4            NaN
2026-06-30        19471.9            NaN
2026-01-07        19406.6            NaN
2026-02-07        19671.8            NaN
2026-03-07        19852.4       0.023985
2026-06-07        19683.8            NaN
2026-07-07        19640.2            NaN
2026-08-07        19104.3            NaN
2026-09-07        19322.8            NaN
2026-10-07        19384.7      -0.015195
2026-07-13      

In [43]:
ibex_df.loc[16, "Date"] = pd.Timestamp("2026-07-01")
ibex_df.loc[17, "Date"] = pd.Timestamp("2026-07-02")
ibex_df.loc[18, "Date"] = pd.Timestamp("2026-07-03")
ibex_df.loc[19, "Date"] = pd.Timestamp("2026-07-06")
ibex_df.loc[20, "Date"] = pd.Timestamp("2026-07-07")
ibex_df.loc[21, "Date"] = pd.Timestamp("2026-07-08")
ibex_df.loc[22, "Date"] = pd.Timestamp("2026-07-09")
ibex_df.loc[23, "Date"] = pd.Timestamp("2026-07-10")

print(ibex_df[["Date", "IBEX 35 Close", "Weekly Return"]].to_string(index=False))

      Date  IBEX 35 Close  Weekly Return
2026-06-10        18142.7            NaN
2026-06-11        18290.1            NaN
2026-06-12        18764.4       0.034267
2026-06-15        19032.0            NaN
2026-06-16        19163.6            NaN
2026-06-17        19421.9            NaN
2026-06-18        19404.1            NaN
2026-06-19        19347.4       0.016572
2026-06-22        19542.3            NaN
2026-06-23        19476.5            NaN
2026-06-24        19389.5            NaN
2026-06-25        19513.6            NaN
2026-06-26        19425.3      -0.005987
2026-06-29        19387.4            NaN
2026-06-30        19471.9            NaN
2026-01-07        19406.6            NaN
2026-07-01        19671.8            NaN
2026-07-02        19852.4       0.023985
2026-07-03        19683.8            NaN
2026-07-06        19640.2            NaN
2026-07-07        19104.3            NaN
2026-07-08        19322.8            NaN
2026-07-09        19384.7      -0.015195
2026-07-10      

In [44]:
print([name for name in globals() if not name.startswith("_")])

['In', 'Out', 'get_ipython', 'exit', 'quit', 'files', 'pd', 'uploaded', 'filename', 'excel_file', 'articles', 'i', 'column', 'df', 'clean_df', 'convert_finance_sentiment', 'positive_terms', 'negative_terms', 'positive_phrases', 'negative_phrases', 're', 'normalize_text', 'find_phrase_hits', 'find_word_hits', 'rule_based_sentiment', 'rule_results', 'rule_df', 'financial_positive_phrases', 'financial_negative_phrases', 'event_based_sentiment', 'event_results', 'event_df', 'os', 'headline_positive_rules', 'headline_negative_rules', 'headline_first_sentiment', 'headline_results', 'headline_df', 'headline_context_rules', 'final_headline_sentiment', 'final_results', 'final_df', 'ibex_df']


In [45]:
# Fix the two incorrectly parsed July dates
ibex_df.loc[
    ibex_df["Date"] == pd.Timestamp("2026-01-07"),
    "Date"
] = pd.Timestamp("2026-07-01")

ibex_df.loc[
    ibex_df["Date"] == pd.Timestamp("2026-07-09"),
    "Date"
] = pd.Timestamp("2026-07-10")

# Sort chronologically
ibex_df = ibex_df.sort_values("Date").reset_index(drop=True)

# Display the cleaned IBEX data
print(ibex_df[["Date", "IBEX 35 Close", "Weekly Return"]].to_string(index=False))


      Date  IBEX 35 Close  Weekly Return
2026-06-10        18142.7            NaN
2026-06-11        18290.1            NaN
2026-06-12        18764.4       0.034267
2026-06-15        19032.0            NaN
2026-06-16        19163.6            NaN
2026-06-17        19421.9            NaN
2026-06-18        19404.1            NaN
2026-06-19        19347.4       0.016572
2026-06-22        19542.3            NaN
2026-06-23        19476.5            NaN
2026-06-24        19389.5            NaN
2026-06-25        19513.6            NaN
2026-06-26        19425.3      -0.005987
2026-06-29        19387.4            NaN
2026-06-30        19471.9            NaN
2026-07-01        19406.6            NaN
2026-07-01        19671.8            NaN
2026-07-02        19852.4       0.023985
2026-07-03        19683.8            NaN
2026-07-06        19640.2            NaN
2026-07-07        19104.3            NaN
2026-07-08        19322.8            NaN
2026-07-10        19384.7      -0.015195
2026-07-10      

In [46]:
# Correct dates, in the exact row order of ibex_df
correct_dates = [
    "10/06/2026",
    "11/06/2026",
    "12/06/2026",
    "15/06/2026",
    "16/06/2026",
    "17/06/2026",
    "18/06/2026",
    "19/06/2026",
    "22/06/2026",
    "23/06/2026",
    "24/06/2026",
    "25/06/2026",
    "26/06/2026",
    "29/06/2026",
    "30/06/2026",
    "01/07/2026",
    "02/07/2026",
    "03/07/2026",
    "06/07/2026",
    "07/07/2026",
    "08/07/2026",
    "09/07/2026",
    "10/07/2026",
    "13/07/2026",
    "14/07/2026",
    "15/07/2026",
    "16/07/2026",
    "17/07/2026",
    "20/07/2026",
    "21/07/2026",
    "22/07/2026",
    "23/07/2026",
    "24/07/2026",
    "27/07/2026",
    "28/07/2026",
    "29/07/2026",
    "30/07/2026",
    "31/07/2026",
    "03/08/2026",
    "04/08/2026",
    "05/08/2026",
    "06/08/2026",
    "07/08/2026",
    "10/08/2026"
]

# Make sure the number of dates matches the number of rows
assert len(correct_dates) == len(ibex_df), (
    f"Expected {len(ibex_df)} dates, but got {len(correct_dates)}"
)

# Replace the entire Date column
ibex_df["Date"] = pd.to_datetime(correct_dates, dayfirst=True)

# Sort chronologically and reset the index
ibex_df = ibex_df.sort_values("Date").reset_index(drop=True)

# Check the result
print(ibex_df[["Date", "IBEX 35 Close", "Weekly Return"]].to_string(index=False))

      Date  IBEX 35 Close  Weekly Return
2026-06-10        18142.7            NaN
2026-06-11        18290.1            NaN
2026-06-12        18764.4       0.034267
2026-06-15        19032.0            NaN
2026-06-16        19163.6            NaN
2026-06-17        19421.9            NaN
2026-06-18        19404.1            NaN
2026-06-19        19347.4       0.016572
2026-06-22        19542.3            NaN
2026-06-23        19476.5            NaN
2026-06-24        19389.5            NaN
2026-06-25        19513.6            NaN
2026-06-26        19425.3      -0.005987
2026-06-29        19387.4            NaN
2026-06-30        19471.9            NaN
2026-07-01        19406.6            NaN
2026-07-02        19671.8            NaN
2026-07-03        19852.4       0.023985
2026-07-06        19683.8            NaN
2026-07-07        19640.2            NaN
2026-07-08        19104.3            NaN
2026-07-09        19322.8            NaN
2026-07-10        19384.7      -0.015195
2026-07-13      

In [39]:
print(final_df[[
    "Date",
    "Title",
    "Final Raw Score",
    "Final Sentiment"
]].to_string(index=False))

      Date                                                                                                                                    Title  Final Raw Score  Final Sentiment
2026-06-10                                                                   La Bolsa cede ante el cóctel de guerra, inflación y ajuste tecnológico               -2             -0.5
2026-06-16                                                                 El Ibex sube un 0,7%, revalida nuevos máximos y supera los 19.100 puntos                7              1.0
2026-06-19                                                            El Ibex gana un 3% en la semana, entre el alivio y la cautela en los mercados                7              1.0
2026-06-22                                                    El precio del petróleo baja a los 77 dólares, niveles del inicio de la guerra en Irán                3              0.5
2026-06-23                                     El mercado refuerza la expectativa de alzas

In [41]:
# Make copies so our original dataframes remain unchanged
article_validation = final_df.copy()
market_validation = ibex_df.copy()

# Make sure dates are proper datetime values
article_validation["Date"] = pd.to_datetime(article_validation["Date"])
market_validation["Date"] = pd.to_datetime(market_validation["Date"])

# Keep only the rows containing the weekly IBEX returns
weekly_ibex = market_validation[
    market_validation["Weekly Return"].notna()
][["Date", "Weekly Return"]].copy()

# Rename the market date so it's clear this is the week-ending date
weekly_ibex = weekly_ibex.rename(
    columns={"Date": "Week Ending"}
)

# Determine the Friday/week-ending date for each article
article_validation["Week Ending"] = (
    article_validation["Date"]
    + pd.to_timedelta(
        4 - article_validation["Date"].dt.weekday,
        unit="D"
    )
)

# Match each article to its week's IBEX return
article_validation = article_validation.merge(
    weekly_ibex,
    on="Week Ending",
    how="left"
)

# Show the result
print(
    article_validation[
        ["Date", "Title", "Final Sentiment", "Week Ending", "Weekly Return"]
    ].to_string(index=False)
)

      Date                                                                                                                                    Title  Final Sentiment Week Ending  Weekly Return
2026-06-10                                                                   La Bolsa cede ante el cóctel de guerra, inflación y ajuste tecnológico             -0.5  2026-06-12            NaN
2026-06-16                                                                 El Ibex sube un 0,7%, revalida nuevos máximos y supera los 19.100 puntos              1.0  2026-06-19       0.016572
2026-06-19                                                            El Ibex gana un 3% en la semana, entre el alivio y la cautela en los mercados              1.0  2026-06-19       0.016572
2026-06-22                                                    El precio del petróleo baja a los 77 dólares, niveles del inicio de la guerra en Irán              0.5  2026-06-26      -0.005987
2026-06-23                              

In [42]:
articles = final_df.copy()
market = ibex_df.copy()

# Convert dates to datetime
articles["Date"] = pd.to_datetime(articles["Date"])
market["Date"] = pd.to_datetime(market["Date"])

# Determine the Friday that ends each article's week
articles["Week Ending"] = (
    articles["Date"]
    + pd.to_timedelta(4 - articles["Date"].dt.weekday, unit="D")
)

# Keep only the IBEX weekly observations
weekly_market = market.loc[
    market["Weekly Return"].notna(),
    ["Date", "Weekly Return"]
].copy()

# Rename the IBEX date so it matches the article's week-ending date
weekly_market = weekly_market.rename(columns={"Date": "Week Ending"})

# Merge without changing either original dataframe
validation_df = articles.merge(
    weekly_market,
    on="Week Ending",
    how="left"
)

print(
    validation_df[
        ["Date", "Final Sentiment", "Week Ending", "Weekly Return"]
    ].to_string(index=False)
)


      Date  Final Sentiment Week Ending  Weekly Return
2026-06-10             -0.5  2026-06-12            NaN
2026-06-16              1.0  2026-06-19       0.016572
2026-06-19              1.0  2026-06-19       0.016572
2026-06-22              0.5  2026-06-26      -0.005987
2026-06-23             -0.5  2026-06-26      -0.005987
2026-06-29             -0.5  2026-07-03            NaN
2026-07-02              1.0  2026-07-03            NaN
2026-07-08             -1.0  2026-07-10            NaN
2026-07-10             -0.5  2026-07-10            NaN
2026-07-17             -0.5  2026-07-17      -0.006144
2026-07-21             -0.5  2026-07-24       0.019706
2026-07-27              0.5  2026-07-31       0.002107
2026-07-28              0.0  2026-07-31       0.002107
2026-07-30              0.5  2026-07-31       0.002107
2026-08-04              0.0  2026-08-07            NaN
2026-08-05              0.5  2026-08-07            NaN
2026-08-07              0.5  2026-08-07            NaN
2026-08-10

In [52]:
print(validation_df.columns.tolist())

['Date', 'Title', 'Final Raw Score', 'Final Sentiment', 'Positive Signals', 'Negative Signals', 'Week Ending', 'Weekly Return']


In [44]:
# Make clean copies
articles = final_df.copy()
market = ibex_df.copy()

# Convert dates to datetime
articles["Date"] = pd.to_datetime(articles["Date"])
market["Date"] = pd.to_datetime(market["Date"])

# Determine the Friday that ends each article's week
articles["Week Ending"] = (
    articles["Date"]
    + pd.to_timedelta(4 - articles["Date"].dt.weekday, unit="D")
)

# Keep only the IBEX weekly observations
weekly_market = market.loc[
    market["Weekly Return"].notna(),
    ["Date", "Weekly Return"]
].copy()

# Rename the IBEX date so it matches the article's week-ending date
weekly_market = weekly_market.rename(columns={"Date": "Week Ending"})

# Merge without changing either original dataframe
validation_df = articles.merge(
    weekly_market,
    on="Week Ending",
    how="left"
)

print(validation_df.columns.tolist())

['Date', 'Title', 'Final Raw Score', 'Final Sentiment', 'Positive Signals', 'Negative Signals', 'Week Ending', 'Weekly Return']


In [54]:
print(
    validation_df[
        ["Date", "Final Sentiment", "Week Ending", "Weekly Return"]
    ].to_string(index=False)
)

      Date  Final Sentiment Week Ending  Weekly Return
2026-06-10             -0.5  2026-06-12       0.034267
2026-06-16              1.0  2026-06-19       0.016572
2026-06-19              1.0  2026-06-19       0.016572
2026-06-22              0.5  2026-06-26      -0.005987
2026-06-23             -0.5  2026-06-26      -0.005987
2026-06-29             -0.5  2026-07-03       0.023985
2026-07-02              1.0  2026-07-03       0.023985
2026-07-08             -1.0  2026-07-10      -0.015195
2026-07-10             -0.5  2026-07-10      -0.015195
2026-07-17             -0.5  2026-07-17      -0.006144
2026-07-21             -0.5  2026-07-24       0.019706
2026-07-27              0.5  2026-07-31       0.002107
2026-07-28              0.0  2026-07-31       0.002107
2026-07-30              0.5  2026-07-31       0.002107
2026-08-04              0.0  2026-08-07       0.009678
2026-08-05              0.5  2026-08-07       0.009678
2026-08-07              0.5  2026-08-07       0.009678
2026-08-10

In [55]:
validation_df = validation_df.dropna(subset=["Weekly Return"]).copy()

print(f"Articles available for validation: {len(validation_df)}")

Articles available for validation: 17


In [56]:
validation_df["Market Direction"] = validation_df["Weekly Return"].apply(
    lambda x: "Positive" if x > 0 else "Negative"
)

print(
    validation_df[
        ["Date", "Final Sentiment", "Weekly Return", "Market Direction"]
    ].to_string(index=False)
)

      Date  Final Sentiment  Weekly Return Market Direction
2026-06-10             -0.5       0.034267         Positive
2026-06-16              1.0       0.016572         Positive
2026-06-19              1.0       0.016572         Positive
2026-06-22              0.5      -0.005987         Negative
2026-06-23             -0.5      -0.005987         Negative
2026-06-29             -0.5       0.023985         Positive
2026-07-02              1.0       0.023985         Positive
2026-07-08             -1.0      -0.015195         Negative
2026-07-10             -0.5      -0.015195         Negative
2026-07-17             -0.5      -0.006144         Negative
2026-07-21             -0.5       0.019706         Positive
2026-07-27              0.5       0.002107         Positive
2026-07-28              0.0       0.002107         Positive
2026-07-30              0.5       0.002107         Positive
2026-08-04              0.0       0.009678         Positive
2026-08-05              0.5       0.0096

In [57]:
validation_df["Predicted Direction"] = validation_df["Final Sentiment"].apply(
    lambda x: "Positive" if x > 0 else ("Negative" if x < 0 else "Neutral")
)

print(
    validation_df[
        ["Date", "Final Sentiment", "Predicted Direction", "Market Direction"]
    ].to_string(index=False)
)

      Date  Final Sentiment Predicted Direction Market Direction
2026-06-10             -0.5            Negative         Positive
2026-06-16              1.0            Positive         Positive
2026-06-19              1.0            Positive         Positive
2026-06-22              0.5            Positive         Negative
2026-06-23             -0.5            Negative         Negative
2026-06-29             -0.5            Negative         Positive
2026-07-02              1.0            Positive         Positive
2026-07-08             -1.0            Negative         Negative
2026-07-10             -0.5            Negative         Negative
2026-07-17             -0.5            Negative         Negative
2026-07-21             -0.5            Negative         Positive
2026-07-27              0.5            Positive         Positive
2026-07-28              0.0             Neutral         Positive
2026-07-30              0.5            Positive         Positive
2026-08-04              0

In [56]:
weekly_sentiment = validation_df.groupby("Week Ending")["Final Sentiment"].mean().reset_index()
weekly_sentiment = weekly_sentiment.rename(columns={"Final Sentiment": "Weekly Sentiment"})

weekly_validation = weekly_sentiment.merge(
    validation_df[["Week Ending", "Weekly Return"]].drop_duplicates(),
    on="Week Ending",
    how="left"
)

print(weekly_validation.to_string(index=False))

Week Ending  Weekly Sentiment  Weekly Return
 2026-06-12         -0.500000            NaN
 2026-06-19          1.000000       0.016572
 2026-06-26          0.000000      -0.005987
 2026-07-03          0.250000            NaN
 2026-07-10         -0.750000            NaN
 2026-07-17         -0.500000      -0.006144
 2026-07-24         -0.500000       0.019706
 2026-07-31          0.333333       0.002107
 2026-08-07          0.333333            NaN
 2026-08-14          0.500000            NaN


In [10]:
from google.colab import files
import pandas as pd

# Upload your Excel workbook
uploaded = files.upload()

# Get the uploaded filename
filename = next(iter(uploaded))

# Show the sheet names
excel_file = pd.ExcelFile(filename)
print("Sheets in workbook:")
print(excel_file.sheet_names)

Saving spanish_economic_news_ibex_2026.xlsx to spanish_economic_news_ibex_2026.xlsx
Sheets in workbook:
['Articles', 'IBEX', 'Weekly Analysis', 'Codebook']


In [61]:
articles = pd.read_excel(filename, sheet_name="Articles")

print("Columns in Articles sheet:")
for i, column in enumerate(articles.columns):
    print(i, ":", column)

Columns in Articles sheet:
0 : Date
1 : Source
2 : Title
3 : URL
4 : Article Text
5 : Sentiment


In [62]:
print(df.columns)
print(df[["Date", "Sentiment"]].to_string(index=False))

Index(['Date', 'Source', 'Title', 'URL', 'Article Text', 'Sentiment'], dtype='object')
      Date  Sentiment
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-10        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-16        NaN
2026-06-19        NaN
2026-06-19        NaN
2026-06-19        NaN
2026-06-19        NaN
2026-06-19 

In [65]:
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"\n{name}")
        print("Columns:", list(obj.columns))
        print("Shape:", obj.shape)


_
Columns: ['Date', 'Title', 'Article Text']
Shape: (5, 3)

articles
Columns: ['Date', 'Source', 'Title', 'URL', 'Article Text', 'Sentiment']
Shape: (337, 6)

df
Columns: ['Date', 'Source', 'Title', 'URL', 'Article Text', 'Sentiment']
Shape: (337, 6)

clean_df
Columns: ['Date', 'Source', 'Title', 'URL', 'Article Text', 'Sentiment']
Shape: (18, 6)

_13
Columns: ['Date', 'Title', 'Article Text']
Shape: (5, 3)

rule_results
Columns: ['Rule Raw Score', 'Rule Sentiment', 'Positive Hits', 'Negative Hits']
Shape: (18, 4)

rule_df
Columns: ['Date', 'Title', 'Rule Raw Score', 'Rule Sentiment', 'Positive Hits', 'Negative Hits']
Shape: (18, 6)

event_results
Columns: ['Event Raw Score', 'Event Sentiment', 'Positive Events', 'Negative Events']
Shape: (18, 4)

event_df
Columns: ['Date', 'Title', 'Event Raw Score', 'Event Sentiment', 'Positive Events', 'Negative Events']
Shape: (18, 6)

headline_results
Columns: ['Headline Raw Score', 'Headline Sentiment', 'Positive Headline Signals', 'Negative Hea

In [45]:
print(final_df[["Date", "Title", "Final Raw Score", "Final Sentiment"]].to_string(index=False))

      Date                                                                                                                                    Title  Final Raw Score  Final Sentiment
2026-06-10                                                                   La Bolsa cede ante el cóctel de guerra, inflación y ajuste tecnológico               -2             -0.5
2026-06-16                                                                 El Ibex sube un 0,7%, revalida nuevos máximos y supera los 19.100 puntos                7              1.0
2026-06-19                                                            El Ibex gana un 3% en la semana, entre el alivio y la cautela en los mercados                7              1.0
2026-06-22                                                    El precio del petróleo baja a los 77 dólares, niveles del inicio de la guerra en Irán                3              0.5
2026-06-23                                     El mercado refuerza la expectativa de alzas

In [47]:
final_df["Date"] = pd.to_datetime(final_df["Date"])

final_df["Week"] = final_df["Date"].dt.to_period("W-SUN")

weekly_raw_sentiment = (
    final_df.groupby("Week")["Final Raw Score"]
    .mean()
    .reset_index()
)

print(weekly_raw_sentiment.to_string(index=False))

                 Week  Final Raw Score
2026-06-08/2026-06-14        -2.000000
2026-06-15/2026-06-21         7.000000
2026-06-22/2026-06-28         0.500000
2026-06-29/2026-07-05         1.000000
2026-07-06/2026-07-12        -6.000000
2026-07-13/2026-07-19        -2.000000
2026-07-20/2026-07-26        -3.000000
2026-07-27/2026-08-02         1.333333
2026-08-03/2026-08-09         1.666667
2026-08-10/2026-08-16         2.000000


In [68]:
raw_weekly = weekly_raw_sentiment.copy()

raw_weekly["Week Ending"] = raw_weekly["Week"].apply(lambda x: x.end_time.date())

raw_weekly = raw_weekly.rename(
    columns={"Final Raw Score": "Weekly Raw Sentiment"}
)

print(raw_weekly.to_string(index=False))

                 Week  Weekly Raw Sentiment Week Ending
2026-06-08/2026-06-14             -2.000000  2026-06-14
2026-06-15/2026-06-21              7.000000  2026-06-21
2026-06-22/2026-06-28              0.500000  2026-06-28
2026-06-29/2026-07-05              1.000000  2026-07-05
2026-07-06/2026-07-12             -6.000000  2026-07-12
2026-07-13/2026-07-19             -2.000000  2026-07-19
2026-07-20/2026-07-26             -3.000000  2026-07-26
2026-07-27/2026-08-02              1.333333  2026-08-02
2026-08-03/2026-08-09              1.666667  2026-08-09
2026-08-10/2026-08-16              2.000000  2026-08-16


In [69]:
complete_raw = raw_weekly[
    (raw_weekly["Week Ending"] >= pd.to_datetime("2026-06-21").date()) &
    (raw_weekly["Week Ending"] <= pd.to_datetime("2026-08-09").date())
].copy()

print(complete_raw.to_string(index=False))
print("\nNumber of complete weeks:", len(complete_raw))


                 Week  Weekly Raw Sentiment Week Ending
2026-06-15/2026-06-21              7.000000  2026-06-21
2026-06-22/2026-06-28              0.500000  2026-06-28
2026-06-29/2026-07-05              1.000000  2026-07-05
2026-07-06/2026-07-12             -6.000000  2026-07-12
2026-07-13/2026-07-19             -2.000000  2026-07-19
2026-07-20/2026-07-26             -3.000000  2026-07-26
2026-07-27/2026-08-02              1.333333  2026-08-02
2026-08-03/2026-08-09              1.666667  2026-08-09

Number of complete weeks: 8


In [71]:
# Get the 8 complete IBEX weekly returns
complete_ibex = weekly_ibex[
    (weekly_ibex["Week Ending"] >= pd.to_datetime("2026-06-21").date()) &
    (weekly_ibex["Week Ending"] <= pd.to_datetime("2026-08-09").date())
].copy()

# Merge raw weekly sentiment with IBEX returns
analysis = pd.merge(
    complete_raw[["Week Ending", "Weekly Raw Sentiment"]],
    complete_ibex[["Week Ending", "Weekly Return"]],
    on="Week Ending",
    how="inner"
)

# Calculate correlation
correlation = analysis["Weekly Raw Sentiment"].corr(
    analysis["Weekly Return"]
)

# Calculate R²
r_squared = correlation ** 2

print(analysis.to_string(index=False))
print("\nNumber of observations:", len(analysis))
print("Correlation (r):", correlation)
print("R²:", r_squared)


TypeError: Invalid comparison between dtype=datetime64[ns] and date

In [72]:
# Make the week-ending dates the same datetime type
complete_ibex = weekly_ibex.copy()
complete_ibex["Week Ending"] = pd.to_datetime(complete_ibex["Week Ending"])

complete_raw2 = complete_raw.copy()
complete_raw2["Week Ending"] = pd.to_datetime(complete_raw2["Week Ending"])

# Keep only the 8 complete weeks
complete_ibex = complete_ibex[
    (complete_ibex["Week Ending"] >= pd.Timestamp("2026-06-21")) &
    (complete_ibex["Week Ending"] <= pd.Timestamp("2026-08-09"))
].copy()

# Merge raw weekly sentiment with IBEX weekly returns
analysis = pd.merge(
    complete_raw2[["Week Ending", "Weekly Raw Sentiment"]],
    complete_ibex[["Week Ending", "Weekly Return"]],
    on="Week Ending",
    how="inner"
)

# Calculate correlation
correlation = analysis["Weekly Raw Sentiment"].corr(
    analysis["Weekly Return"]
)

# Calculate R²
r_squared = correlation ** 2

print(analysis.to_string(index=False))
print("\nNumber of observations:", len(analysis))
print("Correlation (r):", correlation)
print("R²:", r_squared)

Empty DataFrame
Columns: [Week Ending, Weekly Raw Sentiment, Weekly Return]
Index: []

Number of observations: 0
Correlation (r): nan
R²: nan


In [73]:
print("RAW WEEK DATES:")
print(complete_raw2["Week Ending"].to_list())

print("\nIBEX WEEK DATES:")
print(weekly_ibex["Week Ending"].to_list())

print("\nIBEX DATE TYPE:")
print(weekly_ibex["Week Ending"].dtype)

RAW WEEK DATES:
[Timestamp('2026-06-21 00:00:00'), Timestamp('2026-06-28 00:00:00'), Timestamp('2026-07-05 00:00:00'), Timestamp('2026-07-12 00:00:00'), Timestamp('2026-07-19 00:00:00'), Timestamp('2026-07-26 00:00:00'), Timestamp('2026-08-02 00:00:00'), Timestamp('2026-08-09 00:00:00')]

IBEX WEEK DATES:
[Timestamp('2026-06-12 00:00:00'), Timestamp('2026-06-19 00:00:00'), Timestamp('2026-06-26 00:00:00'), Timestamp('2026-07-03 00:00:00'), Timestamp('2026-07-10 00:00:00'), Timestamp('2026-07-17 00:00:00'), Timestamp('2026-07-24 00:00:00'), Timestamp('2026-07-31 00:00:00'), Timestamp('2026-08-07 00:00:00')]

IBEX DATE TYPE:
datetime64[ns]


In [74]:
# Copy the IBEX weekly data
complete_ibex = weekly_ibex.copy()

# Convert the IBEX Friday date into the corresponding
# Monday-Sunday calendar week's Sunday date
complete_ibex["Week Ending"] = (
    pd.to_datetime(complete_ibex["Week Ending"]) +
    pd.Timedelta(days=2)
)

# Make sure raw sentiment dates are datetime
complete_raw2 = complete_raw.copy()
complete_raw2["Week Ending"] = pd.to_datetime(complete_raw2["Week Ending"])

# Merge the two datasets
analysis = pd.merge(
    complete_raw2[["Week Ending", "Weekly Raw Sentiment"]],
    complete_ibex[["Week Ending", "Weekly Return"]],
    on="Week Ending",
    how="inner"
)

print(analysis.to_string(index=False))
print("\nNumber of observations:", len(analysis))

Week Ending  Weekly Raw Sentiment  Weekly Return
 2026-06-21              7.000000       0.016572
 2026-06-28              0.500000      -0.005987
 2026-07-05              1.000000       0.023985
 2026-07-12             -6.000000      -0.015195
 2026-07-19             -2.000000      -0.006144
 2026-07-26             -3.000000       0.019706
 2026-08-02              1.333333       0.002107
 2026-08-09              1.666667       0.009678

Number of observations: 8


In [75]:
# Final correlation and R² using RAW detector-score weekly sentiment

correlation = analysis["Weekly Raw Sentiment"].corr(
    analysis["Weekly Return"]
)

r_squared = correlation ** 2

print("Correlation (r):", correlation)
print("R²:", r_squared)

Correlation (r): 0.5198088977117925
R²: 0.27020129014034877


In [77]:
import numpy as np

x = analysis["Weekly Raw Sentiment"]
y = analysis["Weekly Return"]

slope, intercept = np.polyfit(x, y, 1)

print("Slope:", slope)
print("Intercept:", intercept)
print("R²:", r_squared)

Slope: 0.001902987214949844
Intercept: 0.005471382204690631
R²: 0.27020129014034877


In [38]:
print("Observed minimum Final Raw Score:", final_df["Final Raw Score"].min())
print("Observed maximum Final Raw Score:", final_df["Final Raw Score"].max())

Observed minimum Final Raw Score: -8
Observed maximum Final Raw Score: 7


In [51]:
print("final_df:", final_df.shape)
print("weekly_validation:", weekly_validation.shape)

final_df: (18, 7)
weekly_validation: (10, 3)


In [52]:
display(final_df)

,Date,Title,Final Raw Score,Final Sentiment,Positive Signals,Negative Signals,Week
0,2026-06-10,"La Bolsa cede ante el cóctel de guerra, inflac...",-2,-0.5,,"guerra, inflación",2026-06-08/2026-06-14
1,2026-06-16,"El Ibex sube un 0,7%, revalida nuevos máximos ...",7,1.0,"el ibex sube, nuevos máximos, sube un",,2026-06-15/2026-06-21
2,2026-06-19,"El Ibex gana un 3% en la semana, entre el aliv...",7,1.0,"CONTEXT: alivio, alivio, el ibex gana, gana un",,2026-06-15/2026-06-21
3,2026-06-22,"El precio del petróleo baja a los 77 dólares, ...",3,0.5,"CONTEXT: el petróleo baja, CONTEXT: el precio ...",guerra,2026-06-22/2026-06-28
4,2026-06-23,El mercado refuerza la expectativa de alzas de...,-2,-0.5,,CONTEXT: expectativa de alzas de tipos,2026-06-22/2026-06-28
5,2026-06-29,El Ibex se aleja de sus máximos en un arranque...,-3,-0.5,,se aleja de sus máximos,2026-06-29/2026-07-05
6,2026-07-02,El Ibex toca nuevos máximos al desinflarse las...,5,1.0,"el ibex toca nuevos máximos, nuevos máximos",,2026-06-29/2026-07-05
7,2026-07-08,Sesión negra al anunciar Trump el final del al...,-8,-1.0,,"peor sesión, sesión negra",2026-07-06/2026-07-12
8,2026-07-10,El Ibex registra su peor semana en dos meses m...,-4,-0.5,,peor semana,2026-07-06/2026-07-12
9,2026-07-17,Los inversores cuestionan las valoraciones de ...,-2,-0.5,,CONTEXT: cuestionan las valoraciones,2026-07-13/2026-07-19


In [53]:
github_articles = final_df[
    ["Date", "Title", "Final Raw Score", "Final Sentiment", "Week"]
].copy()

display(github_articles)

,Date,Title,Final Raw Score,Final Sentiment,Week
0,2026-06-10,"La Bolsa cede ante el cóctel de guerra, inflac...",-2,-0.5,2026-06-08/2026-06-14
1,2026-06-16,"El Ibex sube un 0,7%, revalida nuevos máximos ...",7,1.0,2026-06-15/2026-06-21
2,2026-06-19,"El Ibex gana un 3% en la semana, entre el aliv...",7,1.0,2026-06-15/2026-06-21
3,2026-06-22,"El precio del petróleo baja a los 77 dólares, ...",3,0.5,2026-06-22/2026-06-28
4,2026-06-23,El mercado refuerza la expectativa de alzas de...,-2,-0.5,2026-06-22/2026-06-28
5,2026-06-29,El Ibex se aleja de sus máximos en un arranque...,-3,-0.5,2026-06-29/2026-07-05
6,2026-07-02,El Ibex toca nuevos máximos al desinflarse las...,5,1.0,2026-06-29/2026-07-05
7,2026-07-08,Sesión negra al anunciar Trump el final del al...,-8,-1.0,2026-07-06/2026-07-12
8,2026-07-10,El Ibex registra su peor semana en dos meses m...,-4,-0.5,2026-07-06/2026-07-12
9,2026-07-17,Los inversores cuestionan las valoraciones de ...,-2,-0.5,2026-07-13/2026-07-19


In [57]:
github_weekly = weekly_validation[
    ["Week Ending", "Weekly Sentiment", "Weekly Return"]
].copy()

github_weekly = github_weekly[
    (github_weekly["Week Ending"] >= pd.Timestamp("2026-06-21")) &
    (github_weekly["Week Ending"] <= pd.Timestamp("2026-08-09"))
].copy()

github_weekly = github_weekly.rename(columns={
    "Weekly Sentiment": "Weekly Raw Sentiment",
    "Weekly Return": "IBEX Weekly Return"
})

display(github_weekly)

,Week Ending,Weekly Raw Sentiment,IBEX Weekly Return
2,2026-06-26,0.000000,-0.005987
3,2026-07-03,0.250000,NaN
4,2026-07-10,-0.750000,NaN
5,2026-07-17,-0.500000,-0.006144
6,2026-07-24,-0.500000,0.019706
7,2026-07-31,0.333333,0.002107
8,2026-08-07,0.333333,NaN


In [59]:
print(weekly_validation.columns.tolist())
display(weekly_validation)

['Week Ending', 'Weekly Sentiment', 'Weekly Return']


,Week Ending,Weekly Sentiment,Weekly Return
0,2026-06-12,-0.500000,NaN
1,2026-06-19,1.000000,0.016572
2,2026-06-26,0.000000,-0.005987
3,2026-07-03,0.250000,NaN
4,2026-07-10,-0.750000,NaN
5,2026-07-17,-0.500000,-0.006144
6,2026-07-24,-0.500000,0.019706
7,2026-07-31,0.333333,0.002107
8,2026-08-07,0.333333,NaN
9,2026-08-14,0.500000,NaN


In [61]:
github_weekly = pd.DataFrame({
    "Week Ending": pd.to_datetime([
        "2026-06-21",
        "2026-06-28",
        "2026-07-05",
        "2026-07-12",
        "2026-07-19",
        "2026-07-26",
        "2026-08-02",
        "2026-08-09"
    ]),
    "Weekly Raw Sentiment": [
        7.00,
        0.50,
        1.00,
        -6.00,
        -2.00,
        -3.00,
        1.33,
        1.67
    ],
    "IBEX Weekly Return": [
        0.01657208911,
        -0.005987012788,
        0.02398464982,
        -0.01519523669,
        -0.006144075467,
        0.01970645966,
        0.002107257374,
        0.009678420226
    ]
})

display(github_weekly)

,Week Ending,Weekly Raw Sentiment,IBEX Weekly Return
0,2026-06-21,7.00,0.016572
1,2026-06-28,0.50,-0.005987
2,2026-07-05,1.00,0.023985
3,2026-07-12,-6.00,-0.015195
4,2026-07-19,-2.00,-0.006144
5,2026-07-26,-3.00,0.019706
6,2026-08-02,1.33,0.002107
7,2026-08-09,1.67,0.009678


In [63]:
github_articles.to_csv(
    "article_sentiment_analysis.csv",
    index=False
)

github_weekly.to_csv(
    "weekly_sentiment_ibex_analysis.csv",
    index=False
)

print("Files created:")
print("- article_sentiment_analysis.csv")
print("- weekly_sentiment_ibex_analysis.csv")

Files created:
- article_sentiment_analysis.csv
- weekly_sentiment_ibex_analysis.csv


In [64]:
print("ARTICLE DATASET")
print(github_articles.shape)
display(github_articles)

print("\nWEEKLY DATASET")
print(github_weekly.shape)
display(github_weekly)

ARTICLE DATASET
(18, 5)


,Date,Title,Final Raw Score,Final Sentiment,Week
0,2026-06-10,"La Bolsa cede ante el cóctel de guerra, inflac...",-2,-0.5,2026-06-08/2026-06-14
1,2026-06-16,"El Ibex sube un 0,7%, revalida nuevos máximos ...",7,1.0,2026-06-15/2026-06-21
2,2026-06-19,"El Ibex gana un 3% en la semana, entre el aliv...",7,1.0,2026-06-15/2026-06-21
3,2026-06-22,"El precio del petróleo baja a los 77 dólares, ...",3,0.5,2026-06-22/2026-06-28
4,2026-06-23,El mercado refuerza la expectativa de alzas de...,-2,-0.5,2026-06-22/2026-06-28
5,2026-06-29,El Ibex se aleja de sus máximos en un arranque...,-3,-0.5,2026-06-29/2026-07-05
6,2026-07-02,El Ibex toca nuevos máximos al desinflarse las...,5,1.0,2026-06-29/2026-07-05
7,2026-07-08,Sesión negra al anunciar Trump el final del al...,-8,-1.0,2026-07-06/2026-07-12
8,2026-07-10,El Ibex registra su peor semana en dos meses m...,-4,-0.5,2026-07-06/2026-07-12
9,2026-07-17,Los inversores cuestionan las valoraciones de ...,-2,-0.5,2026-07-13/2026-07-19



WEEKLY DATASET
(8, 3)


,Week Ending,Weekly Raw Sentiment,IBEX Weekly Return
0,2026-06-21,7.00,0.016572
1,2026-06-28,0.50,-0.005987
2,2026-07-05,1.00,0.023985
3,2026-07-12,-6.00,-0.015195
4,2026-07-19,-2.00,-0.006144
5,2026-07-26,-3.00,0.019706
6,2026-08-02,1.33,0.002107
7,2026-08-09,1.67,0.009678


In [66]:
readme = r"""# Spanish Economic News Sentiment and IBEX 35 Returns

## Overview

This project examines the relationship between sentiment in Spanish economic news and weekly returns of the IBEX 35 during June–August 2026.

The project developed and tested several approaches to automated sentiment detection before settling on a final headline-focused scoring method. The final detector was applied to 18 selected Spanish economic news articles and produced numerical raw sentiment scores. These scores were aggregated into weekly sentiment measures and compared with IBEX 35 weekly returns across 8 complete weeks.

## Research Question

**Is more positive Spanish economic news sentiment associated with higher weekly IBEX 35 returns?**

## Data

The analysis uses:

- 18 Spanish economic news articles scored by the final sentiment detector.
- IBEX 35 closing-price data for the corresponding trading period.
- 8 complete Monday–Friday trading weeks used for the statistical comparison.

The article-level dataset contains publication dates, article titles, final raw detector scores, final sentiment categories, and calendar-week assignments.

The public dataset does not reproduce full article text.

## Sentiment Detector Development

The detector was developed iteratively:

1. **Rule-based detector** — tested general positive and negative financial language.
2. **Event-based detector** — incorporated economically meaningful events and signals.
3. **Headline-focused detector** — concentrated the scoring on headline-level signals and market-relevant language.
4. **Final headline-first detector** — the final scoring function used for the analysis.

The final detector combines weighted positive and negative headline signals, context-specific financial rules, and direct IBEX movement signals.

The final detector produces a **Final Raw Score** for each article. In the observed dataset, raw scores ranged from **−8 to +7**.

## Sentiment Scale

The Final Raw Score was converted into a five-level sentiment category:

| Raw Score | Final Sentiment |
|---|---:|
| ≤ −5 | −1.0 |
| −4 to −2 | −0.5 |
| −1 to +1 | 0.0 |
| +2 to +4 | +0.5 |
| ≥ +5 | +1.0 |

The categorical Final Sentiment values were manually reviewed as a quality-control step.

Importantly, the **original Final Raw Scores were retained for the weekly quantitative analysis**. The manually validated categorical scores were not substituted for the raw scores.

## Weekly Sentiment

Weekly sentiment was calculated as the arithmetic mean of the Final Raw Scores for articles published during each Monday–Sunday calendar week.

The final market comparison uses 8 complete weeks from **June 15 through August 9, 2026**.

The incomplete June 8–14 and August 10–16 weeks were excluded from the market comparison.

## IBEX 35 Weekly Return

For each complete week, the IBEX 35 weekly return was calculated as:

**(Friday Close − Monday Close) / Monday Close × 100**

This Monday-to-Friday definition was applied consistently across all 8 weeks.

## Results

The final analysis contains **8 weekly observations**.

- **Pearson correlation (r): 0.5198**
- **R²: 0.2702**
- **Regression slope: 0.001903**
- **Regression intercept: 0.005471**

The positive correlation indicates a **moderate positive linear association** between weekly raw news sentiment and weekly IBEX 35 returns in this sample.

The R² of approximately 0.27 indicates that about 27% of the variation in weekly IBEX returns is associated with the linear relationship with weekly sentiment in this sample.

### Important interpretation

The results show **association, not causation**. The analysis does not establish that changes in news sentiment caused changes in the IBEX 35.

## Limitations

The analysis has several important limitations:

- The statistical sample contains only 8 complete weekly observations.
- The underlying article sample contains 18 scored articles.
- The sentiment detector is a rule-based, headline-focused approach rather than a trained machine-learning model.
- Detector scores depend on the selected financial language rules and weighting system.
- Manual validation provides a quality-control check but does not eliminate subjectivity.
- Other factors may influence IBEX 35 returns, including macroeconomic announcements, geopolitical developments, monetary policy expectations, company-specific news, and international market movements.

Therefore, the results should be interpreted as an exploratory analysis rather than evidence of a stable predictive relationship.

## Repository Structure

```text
spanish-economic-news-ibex-sentiment/
├── README.md
├── data/
│   ├── article_sentiment_analysis.csv
│   └── weekly_sentiment_ibex_analysis.csv
├── analysis/
│   └── sentiment_ibex_analysis.ipynb
└── figures/
    ├── graph1_weekly_sentiment.png
    ├── graph2_ibex_returns.png
    └── graph3_sentiment_vs_ibex.png

SyntaxError: incomplete input (90662899.py, line 1)

In [67]:
readme = """# Spanish Economic News Sentiment and IBEX 35 Returns

## Overview

This project examines the relationship between sentiment in Spanish economic news and weekly returns of the IBEX 35 during June-August 2026.

The final headline-focused sentiment detector was applied to 18 Spanish economic news articles. The resulting raw sentiment scores were aggregated into weekly measures and compared with IBEX 35 weekly returns across 8 complete weeks.

## Research Question

Is more positive Spanish economic news sentiment associated with higher weekly IBEX 35 returns?

## Final Results

- Observations: 8 complete weeks
- Pearson correlation (r): 0.5198
- R-squared: 0.2702
- Regression slope: 0.001903
- Regression intercept: 0.005471

The results indicate a moderate positive linear association in this sample. The analysis does not establish causation.

## Methodology

The sentiment detector was developed iteratively through four stages:

1. Rule-based detector
2. Event-based detector
3. Headline-focused detector
4. Final headline-first scoring function

The final detector produces a numerical Final Raw Score for each article. Observed scores ranged from -8 to +7.

Raw scores were converted into five sentiment categories:

- -5 or below = -1.0
- -4 to -2 = -0.5
- -1 to +1 = 0.0
- +2 to +4 = +0.5
- +5 or above = +1.0

The categorical scores were manually reviewed for quality control. However, the original Final Raw Scores were retained for the weekly quantitative analysis.

Weekly sentiment was calculated as the mean Final Raw Score for articles published during each Monday-Sunday calendar week.

IBEX 35 weekly return was calculated as:

(Friday Close - Monday Close) / Monday Close * 100

The final comparison contains 8 complete weeks from June 15 through August 9, 2026.

## Limitations

The sample contains only 18 scored articles and 8 complete weekly observations. The detector is a rule-based, headline-focused approach rather than a trained machine-learning model.

The correlation measures association rather than causation. Other factors, including macroeconomic developments, geopolitical events, monetary policy expectations, company-specific news, and international markets may also affect IBEX 35 returns.

Therefore, the findings should be considered exploratory rather than evidence of a stable predictive relationship.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("README.md created successfully.")

README.md created successfully.


In [68]:
import os

os.makedirs("spanish-economic-news-ibex-sentiment/data", exist_ok=True)
os.makedirs("spanish-economic-news-ibex-sentiment/analysis", exist_ok=True)
os.makedirs("spanish-economic-news-ibex-sentiment/figures", exist_ok=True)

print("Repository folders created successfully.")

Repository folders created successfully.


In [69]:
import shutil
import os

# Copy README
shutil.copy(
    "README.md",
    "spanish-economic-news-ibex-sentiment/README.md"
)

# Copy verified datasets
shutil.copy(
    "article_sentiment_analysis.csv",
    "spanish-economic-news-ibex-sentiment/data/article_sentiment_analysis.csv"
)

shutil.copy(
    "weekly_sentiment_ibex_analysis.csv",
    "spanish-economic-news-ibex-sentiment/data/weekly_sentiment_ibex_analysis.csv"
)

print("README and both verified datasets copied successfully.")

README and both verified datasets copied successfully.


In [70]:
import os

for root, dirs, files in os.walk("spanish-economic-news-ibex-sentiment"):
    level = root.replace("spanish-economic-news-ibex-sentiment", "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}    {file}")

spanish-economic-news-ibex-sentiment/
    README.md
    analysis/
    figures/
    data/
        article_sentiment_analysis.csv
        weekly_sentiment_ibex_analysis.csv


In [71]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles:")
for f in os.listdir("."):
    print(f)

Current directory:
/content

Files:
.config
spanish_economic_news_ibex_2026.xlsx
article_sentiment_analysis.csv
README.md
spanish-economic-news-ibex-sentiment
weekly_sentiment_ibex_analysis.csv
sample_data
